# Descriptive statistics table for treated and control municipalities

This notebook loads the analysis-ready municipality-quarter panel from `ROOT/out`, constructs a pooled **treated vs control** grouping, and creates a descriptive-statistics table with these rows:

- Average Rent Price Pre
- Average Rent Price Post
- Yearly Rent Growth (2011-2025)
- Yearly Rent Growth (2022-2024)
- Average Rent Growth Post
- Tenancy Agreements Pre
- Tenancy Agreements Post
- Rent Price/Taxable Base
- Yearly Taxable Base
- Population
- Natives
- Foreigners
- Net Inflows
- Net Outflows
- Number of Municipalities
- Municipalities with Data on Rents

**Important choice:** for this pooled descriptive table, `Pre`/`Post` are defined relative to the **first implementation quarter**, with:

- `Pre`: before **2024-Q2**
- `Post`: from **2024-Q2** onward

That is a descriptive convention for the table, not the staggered-treatment estimator itself.


In [1]:
from pathlib import Path
import json
import ast
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PKL_PATH = OUT_DIR / "analysis_ready_panel.pkl"
JSONL_PATH = OUT_DIR / "analysis_ready_panel.jsonl"

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("analysis_ready_panel.pkl exists:", PKL_PATH.exists())
print("analysis_ready_panel.jsonl exists:", JSONL_PATH.exists())


ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
analysis_ready_panel.pkl exists: True
analysis_ready_panel.jsonl exists: True


In [3]:
if PKL_PATH.exists():
    panel = pd.read_pickle(PKL_PATH)
    source_used = PKL_PATH
elif JSONL_PATH.exists():
    panel = pd.read_json(JSONL_PATH, orient="records", lines=True)
    source_used = JSONL_PATH
else:
    raise FileNotFoundError(
        "Could not find analysis_ready_panel.pkl or analysis_ready_panel.jsonl in ROOT/out"
    )

print("Loaded from:", source_used)
print("Shape:", panel.shape)
panel.head()


Loaded from: /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.pkl
Shape: (56908, 51)


,Code,Municipality,municipality_norm,Year,Quarter,quarter_num,time_index,Prices,Contracts,Treated 1,Treated 2,Control,ever_treated,cohort,first_treat_period,first_treat_index,treated,event_time,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males,pop_foreign_females,mig_total,mig_total_males,mig_total_females,mig_inflows,mig_inflows_males,mig_inflows_females,mig_outflows,mig_outflows_males,mig_outflows_females,income_tax_base,income_tax_liability,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,log_price_growth_qoq,log_price_growth_yoy,contracts_per_10k,contracts_per_10k_qoq_pct,contracts_per_10k_yoy_pct,rent_to_income_proxy,annualized_rent_to_income_proxy,rent_to_tax_liability_proxy,outcome_price_growth,outcome_market_activity
0,00005,NaN,,2015,Q1,1,8061,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,,2015,Q2,2,8062,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,,2015,Q3,3,8063,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,,2015,Q4,4,8064,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,,2011,Q1,1,8045,NaN,NaN,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
required_cols = [
    "Code", "Municipality", "Year", "Quarter",
    "Prices", "Contracts",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base", "rent_to_income_proxy",
    "price_growth_yoy_pct",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns in analysis_ready_panel: {missing_required}")

# Build the pooled group indicator
if "ever_treated" in panel.columns:
    panel["Group"] = np.where(panel["ever_treated"].fillna(0).astype(int) == 1, "Treated", "Control")
elif {"Treated 1", "Treated 2"}.issubset(panel.columns):
    ever_treated = (
        panel["Treated 1"].fillna(0).astype(int) == 1
    ) | (
        panel["Treated 2"].fillna(0).astype(int) == 1
    )
    panel["Group"] = np.where(ever_treated, "Treated", "Control")
else:
    raise ValueError("Could not identify treated vs control groups from the loaded panel.")

quarter_num_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
panel["quarter_num"] = panel["Quarter"].map(quarter_num_map)
panel["time_index"] = panel["Year"].astype(int) * 4 + panel["quarter_num"].astype(int)

POST_START_YEAR = 2024
POST_START_QUARTER = "Q2"
POST_START_INDEX = POST_START_YEAR * 4 + quarter_num_map[POST_START_QUARTER]

panel["is_pre"] = panel["time_index"] < POST_START_INDEX
panel["is_post"] = panel["time_index"] >= POST_START_INDEX

# Make sure numeric columns are numeric
numeric_cols = [
    "Prices", "Contracts",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base", "rent_to_income_proxy",
    "price_growth_yoy_pct",
]
for c in numeric_cols:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

panel[["Code", "Municipality", "Year", "Quarter", "Group", "is_pre", "is_post"]].head(12)


,Code,Municipality,Year,Quarter,Group,is_pre,is_post
0,00005,NaN,2015,Q1,Control,True,False
1,00005,NaN,2015,Q2,Control,True,False
2,00005,NaN,2015,Q3,Control,True,False
3,00005,NaN,2015,Q4,Control,True,False
4,02050,NaN,2011,Q1,Control,True,False
5,02050,NaN,2011,Q2,Control,True,False
6,02050,NaN,2011,Q3,Control,True,False
7,02050,NaN,2011,Q4,Control,True,False
8,02050,NaN,2012,Q1,Control,True,False
9,02050,NaN,2012,Q2,Control,True,False


## Table construction strategy

For continuous variables, the table uses **equal municipality weighting**:

1. First compute each municipality's mean over the relevant sample window.
2. Then average those municipality-level means within `Treated` and `Control`.

This prevents municipalities with more valid rows from receiving extra weight.

For counts:
- `Number of Municipalities` is the number of unique municipalities in each group.
- `Municipalities with Data on Rents` is the number of unique municipalities with at least one non-missing `Prices` observation.


In [5]:
def municipality_mean_by_group(df: pd.DataFrame, value_col: str, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    tmp = tmp[["Code", "Group", value_col]].copy()
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")
    tmp = tmp.dropna(subset=[value_col])

    if tmp.empty:
        return pd.Series({"Treated": np.nan, "Control": np.nan}, dtype=float)

    muni_means = (
        tmp.groupby(["Code", "Group"], as_index=False)[value_col]
        .mean()
    )

    group_means = muni_means.groupby("Group")[value_col].mean()

    return pd.Series({
        "Treated": group_means.get("Treated", np.nan),
        "Control": group_means.get("Control", np.nan),
    })


def municipality_count_by_group(df: pd.DataFrame, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    counts = tmp.groupby("Group")["Code"].nunique()

    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })


def municipality_count_with_nonmissing(df: pd.DataFrame, value_col: str, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")
    tmp = tmp[tmp[value_col].notna()].copy()

    counts = tmp.groupby("Group")["Code"].nunique()

    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })


In [6]:
stats_rows = {
    "Average Rent Price Pre": municipality_mean_by_group(panel, "Prices", panel["is_pre"]),
    "Average Rent Price Post": municipality_mean_by_group(panel, "Prices", panel["is_post"]),
    "Yearly Rent Growth (2011-2025)": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        panel["Year"].between(2011, 2025),
    ),
    "Yearly Rent Growth (2022-2024)": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        panel["Year"].between(2022, 2024),
    ),
    "Average Rent Growth Post": municipality_mean_by_group(
        panel,
        "price_growth_yoy_pct",
        panel["is_post"],
    ),
    "Tenancy Agreements Pre": municipality_mean_by_group(panel, "Contracts", panel["is_pre"]),
    "Tenancy Agreements Post": municipality_mean_by_group(panel, "Contracts", panel["is_post"]),
    "Rent Price/Taxable Base": municipality_mean_by_group(panel, "rent_to_income_proxy"),
    "Yearly Taxable Base": municipality_mean_by_group(panel, "income_tax_base"),
    "Population": municipality_mean_by_group(panel, "pop_total"),
    "Natives": municipality_mean_by_group(panel, "pop_native_total"),
    "Foreigners": municipality_mean_by_group(panel, "pop_foreign_total"),
    "Net Inflows": municipality_mean_by_group(panel, "mig_inflows"),
    "Net Outflows": municipality_mean_by_group(panel, "mig_outflows"),
    "Number of Municipalities": municipality_count_by_group(panel),
    "Municipalities with Data on Rents": municipality_count_with_nonmissing(panel, "Prices"),
}

desc_table = pd.DataFrame(stats_rows).T.reset_index().rename(columns={"index": "Statistic"})
desc_table = desc_table[["Statistic", "Treated", "Control"]]

desc_table


,Statistic,Treated,Control
0,Average Rent Price Pre,482.6192,44.5759
1,Average Rent Price Post,665.8008,75.6669
2,Yearly Rent Growth (2011-2025),inf,inf
3,Yearly Rent Growth (2022-2024),inf,inf
4,Average Rent Growth Post,inf,inf
5,Tenancy Agreements Pre,129.7083,2.7554
6,Tenancy Agreements Post,94.5836,2.7037
7,Rent Price/Taxable Base,0.0213,0.0021
8,Yearly Taxable Base,"22,847.3680","19,749.9643"
9,Population,"25,624.9658","1,062.9853"


In [ ]:
# Optional formatting for presentation / export
count_rows = {
    "Number of Municipalities",
    "Municipalities with Data on Rents",
}

desc_table_display = desc_table.copy()
for col in ["Treated", "Control"]:
    desc_table_display[col] = desc_table_display.apply(
        lambda r: int(round(r[col])) if r["Statistic"] in count_rows and pd.notna(r[col]) else r[col],
        axis=1,
    )

desc_table_display


In [ ]:
OUT_CSV = OUT_DIR / "descriptive_stats_treated_control.csv"
OUT_XLSX = OUT_DIR / "descriptive_stats_treated_control.xlsx"
OUT_LATEX = OUT_DIR / "descriptive_stats_treated_control.tex"

desc_table_display.to_csv(OUT_CSV, index=False)
desc_table_display.to_excel(OUT_XLSX, index=False)

# Simple LaTeX export for paper drafting
latex_table = desc_table_display.to_latex(index=False, escape=False, float_format="%.4f")
OUT_LATEX.write_text(latex_table, encoding="utf-8")

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_XLSX)
print(" -", OUT_LATEX)


In [ ]:
# Quick diagnostics
print("Unique municipalities by group:")
display(
    panel.groupby("Group")["Code"].nunique().rename("n_municipalities").reset_index()
)

print("Share of non-missing Prices by group:")
display(
    panel.assign(has_price=panel["Prices"].notna().astype(int))
         .groupby("Group", as_index=False)["has_price"]
         .mean()
)
